# 05. Human-in-the-Loop & Performance Monitoring

This notebook demonstrates the user feedback loop (Feedback Override), model performance monitoring, and the Active Learning data flywheel (extracting hard cases for retraining). This process runs independently and acts as a sandbox to prove the MLOps pipeline concept.


In [ ]:
# ==========================================
# CELL 1: IMPORTS & SETUP
# ==========================================
import sys
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import random

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Import existing inference pipeline
from grading.src.grading.inference import predict

# Paths for Feedback System
FEEDBACK_FILE = PROJECT_ROOT / "grading" / "data" / "feedback_log.csv"
HARD_CASES_DIR = PROJECT_ROOT / "grading" / "data" / "hard_cases"
RAW_DIR = PROJECT_ROOT / "grading" / "data" / "raw" / "Fruit And Vegetable Diseases Dataset"

HARD_CASES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup completed. MLOps Directories:")
print(f"- Feedback Log : {FEEDBACK_FILE}")
print(f"- Hard Cases   : {HARD_CASES_DIR}")


## 1. Feedback Logging System
Creates a simple system to record when a user manually overrides the AI's prediction (both the Grade and the Fruit Type).


In [ ]:
# ==========================================
# CELL 2: FEEDBACK LOGGER CLASS
# ==========================================
class FeedbackSystem:
    def __init__(self, log_file):
        self.log_file = log_file
        # Create CSV file with headers if it doesn't exist
        if not self.log_file.exists():
            df = pd.DataFrame(columns=[
                "timestamp", "image_name", "ai_fruit", "user_fruit", 
                "ai_grade", "user_grade", "override_reason"
            ])
            df.to_csv(self.log_file, index=False)
            
    def log_override(self, image_path, ai_fruit, user_fruit, ai_grade, user_grade, reason):
        """Save a user override to the database/csv"""
        img_name = Path(image_path).name
        new_row = pd.DataFrame([{
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "image_name": img_name,
            "ai_fruit": ai_fruit,
            "user_fruit": user_fruit,
            "ai_grade": ai_grade,
            "user_grade": user_grade,
            "override_reason": reason
        }])
        
        # Append to CSV
        new_row.to_csv(self.log_file, mode='a', header=False, index=False)
        print(f"\n[LOGGED] Record saved for {img_name}:")
        print(f"  - Fruit: '{ai_fruit}' -> '{user_fruit}'")
        print(f"  - Grade: '{ai_grade}' -> '{user_grade}'")

logger = FeedbackSystem(FEEDBACK_FILE)


## 2. Interactive Manual Override
Run the cell below to test an image with the AI, and then use the input boxes to simulate a user overriding the result.


In [ ]:
# ==========================================
# CELL 3: INTERACTIVE DETECTION & OVERRIDE
# ==========================================
# Change this path to test a different image
test_image_path = RAW_DIR / "Tomato__Rotten" / "Tomato_rotten_001.jpg" # Update with a valid filename if this doesn't exist

def interactive_override_test(img_path):
    if not img_path.exists():
        # Fallback to finding a random image if the hardcoded one doesn't exist
        all_imgs = list(RAW_DIR.glob("*/*.jpg"))
        if not all_imgs:
            print("No images found in dataset directory!")
            return
        img_path = random.choice(all_imgs)
        print(f"Fallback: Picked random image {img_path.name}")
        
    print(f"Analyzing: {img_path.name}...")
    
    # 1. AI Inference
    result = predict(str(img_path))
    if not result.get('success', False):
        print("Detection failed:", result.get('error'))
        return
        
    ai_grade = result['grade']
    fruit_class = result.get('class_name', 'Unknown').split('_')[0] # Get just the fruit name, e.g., "Tomato" from "Tomato_Healthy"
    
    # 2. Display Image and Result
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title(f"AI Prediction: {ai_grade}\nDetected Class: {fruit_class}", fontweight='bold', color='darkred' if 'F' in ai_grade else 'darkgreen')
    plt.axis('off')
    plt.show()
    
    # 3. Simulate User Feedback Loop
    print("-" * 50)
    print("USER OVERRIDE SIMULATION")
    print("-" * 50)
    override_decision = input(f"AI predicted '{fruit_class}' with '{ai_grade}'. Do you want to override this result? (y/n): ")
    
    if override_decision.lower() == 'y':
        new_fruit = input(f"Enter correct fruit type (Press Enter to keep '{fruit_class}'): ")
        if not new_fruit.strip():
            new_fruit = fruit_class
            
        new_grade = input(f"Enter new grade (Press Enter to keep '{ai_grade}'): ")
        if not new_grade.strip():
            new_grade = ai_grade
            
        reason = input("Enter reason for override (e.g., 'Lighting was too dark', 'Wrong fruit'): ")
        
        # Log to system
        logger.log_override(img_path, fruit_class, new_fruit, ai_grade, new_grade, reason)
        print("\nSuccess! Feedback has been recorded into the database.")
    else:
        print("\nUser accepted the AI's prediction. No override logged.")

# Run the interactive test
interactive_override_test(test_image_path)


## 3. Simulate Batch Data
Generates automated dummy logs so our monitoring dashboard has enough data to display meaningful charts.


In [ ]:
# ==========================================
# CELL 4: SIMULATE 1 MONTH OF USER FEEDBACK
# ==========================================
def generate_dummy_data(n_records=100):
    fruits = ["Apple", "Tomato", "Mango", "Banana", "Orange", "Guava"]
    ai_grades = ["Grade A", "Grade B", "Grade C", "Grade F (Rotten/Defective)"]
    
    print(f"Generating {n_records} simulated feedback records for the dashboard...")
    
    for i in range(n_records):
        fruit = random.choice(fruits)
        
        # Simulate Model Drift (e.g., AI often mistakes healthy Tomatoes as Rotten due to lighting)
        if fruit == "Tomato" and random.random() < 0.6:
            ai_grade = "Grade F (Rotten/Defective)"
            user_grade = random.choice(["Grade B", "Grade C"])
            user_fruit = fruit
            reason = "Poor lighting caused AI to detect a false defect"
        elif random.random() < 0.1:
            # Simulate misclassification of the fruit itself
            user_fruit = random.choice([f for f in fruits if f != fruit])
            ai_grade = "Grade F (Rotten/Defective)"
            user_grade = "Grade A"
            reason = "AI misclassified the fruit type entirely"
        else:
            ai_grade = random.choice(ai_grades)
            # Users typically upgrade 1 level when overriding
            user_grade = "Grade A" if ai_grade in ["Grade B", "Grade C"] else "Grade B"
            user_fruit = fruit
            reason = "Fruit is still fresh, AI scoring was too strict"
            
        img_name = f"simulated_{fruit.lower()}_{i}.jpg"
        # Supress printing for dummy generation
        new_row = pd.DataFrame([{
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "image_name": img_name,
            "ai_fruit": fruit,
            "user_fruit": user_fruit,
            "ai_grade": ai_grade,
            "user_grade": user_grade,
            "override_reason": reason
        }])
        new_row.to_csv(FEEDBACK_FILE, mode='a', header=False, index=False)
        
    print("Dummy data generation complete!")

# Run to populate database
generate_dummy_data(50)


## 4. Performance Monitoring Dashboard
Visualizes the Override Rate to detect which classes are suffering from degraded performance (Model Drift) or misclassification.


In [ ]:
# ==========================================
# CELL 5: MONITORING DASHBOARD
# ==========================================
def plot_monitoring_dashboard():
    if not FEEDBACK_FILE.exists():
        print("No feedback data found.")
        return
        
    df = pd.read_csv(FEEDBACK_FILE)
    
    if len(df) == 0:
        print("Feedback database is empty.")
        return
        
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Overrides by Fruit Type
    fruit_counts = df['user_fruit'].value_counts()
    fruit_counts.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
    axes[0].set_title("Total Overrides by Fruit Type", fontweight='bold')
    axes[0].set_ylabel("Number of User Corrections")
    axes[0].tick_params(axis='x', rotation=45)
    
    # Plot 2: Misclassifications (AI Fruit != User Fruit)
    misclassified = df[df['ai_fruit'] != df['user_fruit']]
    if len(misclassified) > 0:
        misclass_counts = misclassified['ai_fruit'].value_counts()
        misclass_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', cmap='Set3')
        axes[1].set_title("Fruit Misclassifications (Wrong Detection)", fontweight='bold')
        axes[1].set_ylabel("")
    else:
        axes[1].text(0.5, 0.5, "No Misclassifications", ha='center', va='center')
        axes[1].set_title("Fruit Misclassifications")
        
    plt.tight_layout()
    plt.show()
    
    # Display Top 5 Reasons
    print("\n=== TOP REASONS FOR OVERRIDE ===")
    print(df['override_reason'].value_counts().head(5))

plot_monitoring_dashboard()


## 5. Active Learning (Data Flywheel)
Automatically extracts the misclassified images and moves them to a special directory for the AI Engineers to use in the next training cycle (Fine-tuning).


In [ ]:
# ==========================================
# CELL 6: DATA FLYWHEEL (EXTRACT HARD CASES)
# ==========================================
def extract_hard_cases_for_retraining():
    df = pd.read_csv(FEEDBACK_FILE)
    
    # Filter critical cases (Wrong fruit type or False Positives for defects)
    hard_cases = df[(df['ai_grade'] == 'Grade F (Rotten/Defective)') | (df['ai_fruit'] != df['user_fruit'])]
    
    print(f"Found {len(hard_cases)} critical hard cases.")
    print(f"Target directory: {HARD_CASES_DIR}")
    
    saved_count = 0
    for idx, row in hard_cases.iterrows():
        img_name = row['image_name']
        fruit = row['user_fruit'] # Use the user's corrected fruit type for saving
        
        # Simulate copying the file to the retraining folder
        dummy_file = HARD_CASES_DIR / f"HARD_{fruit}_{img_name}.txt"
        with open(dummy_file, 'w', encoding='utf-8') as f:
            f.write(f"Original AI Fruit: {row['ai_fruit']} -> User Corrected: {row['user_fruit']}\n")
            f.write(f"Original AI Grade: {row['ai_grade']} -> User Corrected: {row['user_grade']}\n")
            f.write(f"Reason: {row['override_reason']}")
        saved_count += 1
        
    print(f"\n[SUCCESS] Exported {saved_count} items to the Active Learning pipeline.")
    print("These images will be reviewed and added to the Kaggle dataset for the next YOLO fine-tuning phase.")

extract_hard_cases_for_retraining()
